# Baseline closure model (F2-only)

Builds a per-institution feature matrix from `processed_data/f2_labeled.csv` and fits two baselines (logistic regression, gradient-boosted trees) against `closed_by_2024`.

**Cohort design (prospective, leak-safe):**
- `AS_OF = 2019`. Population is institutions reporting F2 in 2019 (i.e. demonstrably alive at the as-of date).
- Features come from a 3-year look-back window `[2017, 2018, 2019]`. No data after the as-of date is used.
- Label is `closed_by_2024` measured on the cohort. Because every institution in the cohort was alive at `AS_OF`, any closure happens strictly *after* the feature window, so there is no survivorship leakage from "missing recent years."

In [36]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PROCESSED = ROOT / "processed_data"

AS_OF = 2019
WINDOW = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

labeled = pd.read_csv(PROCESSED / "f2_labeled.csv", low_memory=False)
labeled.shape

(20184, 32)

In [37]:
# Cohort: institutions with an F2 row at the as-of year.
alive_at_as_of = set(labeled.loc[labeled.year == AS_OF, "UNITID"].unique())
panel = labeled[labeled.UNITID.isin(alive_at_as_of) & labeled.year.isin(WINDOW)].copy()

NUM = [
    "F2D01", "F2D16", "F2A02", "F2A03",
    "total_expenses", "change_in_net_assets", "expendable_net_assets",
    "endowment_eoy",
    "operating_margin", "expenses_to_revenue", "assets_to_liabilities",
]
for c in NUM:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

label = (
    labeled[labeled.UNITID.isin(alive_at_as_of)]
    .groupby("UNITID")["closed_by_2024"].max().astype(int)
)
print(f"cohort size: {len(label)}  positive rate: {label.mean():.4f}  positives: {int(label.sum())}")

cohort size: 1836  positive rate: 0.0714  positives: 131


In [38]:
# Per-institution feature build over the look-back window.
# Three feature families: most-recent value, 3-year mean, and 3-year trend.
panel_sorted = panel.sort_values(["UNITID", "year"])
last = (
    panel_sorted.groupby("UNITID")[NUM].last()
    .add_suffix("__last")
)

MEAN_FIELDS = ["operating_margin", "expenses_to_revenue", "assets_to_liabilities", "reported_field_count"]
mean3 = panel.groupby("UNITID")[MEAN_FIELDS].mean().add_suffix("__mean3")

TREND_FIELDS = ["F2D16", "total_expenses", "expendable_net_assets"]

def slope(g: pd.DataFrame, col: str) -> float:
    s = g[["year", col]].dropna()
    if len(s) < 2:
        return np.nan
    y = s[col].to_numpy()
    if np.std(y) == 0:
        return 0.0
    return np.polyfit(s["year"].to_numpy(), y, 1)[0]

def pct_change_window(g: pd.DataFrame, col: str) -> float:
    s = g[["year", col]].dropna().sort_values("year")
    if len(s) < 2 or s[col].iloc[0] == 0:
        return np.nan
    return (s[col].iloc[-1] - s[col].iloc[0]) / abs(s[col].iloc[0])

slopes = panel.groupby("UNITID").apply(
    lambda g: pd.Series({f"{c}__slope": slope(g, c) for c in TREND_FIELDS}),
    include_groups=False,
)
pct = panel.groupby("UNITID").apply(
    lambda g: pd.Series({f"{c}__pct3": pct_change_window(g, c) for c in TREND_FIELDS}),
    include_groups=False,
)

feat = last.join(mean3).join(slopes).join(pct)

# Sign-preserving log transforms for big-magnitude stocks/flows.
for c in ["F2D16", "F2A02", "F2A03", "total_expenses", "expendable_net_assets", "endowment_eoy"]:
    s = feat[f"{c}__last"]
    feat[f"log_{c}"] = np.sign(s) * np.log1p(np.abs(s))

feat["tuition_dependence"] = feat["F2D01__last"] / feat["F2D16__last"].replace({0: np.nan})

feat = feat.replace([np.inf, -np.inf], np.nan)
y = label.reindex(feat.index)
feat.shape, feat.columns.tolist()

((1836, 28),
 ['F2D01__last',
  'F2D16__last',
  'F2A02__last',
  'F2A03__last',
  'total_expenses__last',
  'change_in_net_assets__last',
  'expendable_net_assets__last',
  'endowment_eoy__last',
  'operating_margin__last',
  'expenses_to_revenue__last',
  'assets_to_liabilities__last',
  'operating_margin__mean3',
  'expenses_to_revenue__mean3',
  'assets_to_liabilities__mean3',
  'reported_field_count__mean3',
  'F2D16__slope',
  'total_expenses__slope',
  'expendable_net_assets__slope',
  'F2D16__pct3',
  'total_expenses__pct3',
  'expendable_net_assets__pct3',
  'log_F2D16',
  'log_F2A02',
  'log_F2A03',
  'log_total_expenses',
  'log_expendable_net_assets',
  'log_endowment_eoy',
  'tuition_dependence'])

In [39]:
# 5-fold stratified CV for two baselines.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
gbt = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clf", GradientBoostingClassifier(random_state=42)),
])

results = {}
for name, model in [("logreg", logreg), ("gbt", gbt)]:
    auc = cross_val_score(model, feat, y, cv=cv, scoring="roc_auc")
    ap = cross_val_score(model, feat, y, cv=cv, scoring="average_precision")
    results[name] = {"auc_mean": auc.mean(), "auc_std": auc.std(), "ap_mean": ap.mean(), "ap_std": ap.std()}

pd.DataFrame(results).T.round(3)

,auc_mean,auc_std,ap_mean,ap_std
logreg,0.848,0.026,0.370,0.081
gbt,0.859,0.034,0.417,0.086


In [40]:
# GBT feature importances (fit on the full cohort).
gbt.fit(feat, y)
importances = (
    pd.Series(gbt.named_steps["clf"].feature_importances_, index=feat.columns)
    .sort_values(ascending=False)
)
importances.head(15)

F2D16__slope                    0.093100
tuition_dependence              0.080067
expenses_to_revenue__last       0.076191
total_expenses__pct3            0.072594
operating_margin__last          0.067507
assets_to_liabilities__last     0.065238
operating_margin__mean3         0.054017
F2A02__last                     0.052210
expendable_net_assets__pct3     0.044487
assets_to_liabilities__mean3    0.041285
log_F2A02                       0.040877
F2D16__pct3                     0.037220
F2D01__last                     0.034356
log_endowment_eoy               0.033862
log_F2D16                       0.030404
dtype: float64

In [41]:
# Logistic-regression coefficients (signed direction of effect after standardization).
logreg.fit(feat, y)
lr_coefs = (
    pd.Series(logreg.named_steps["clf"].coef_.ravel(), index=feat.columns)
    .sort_values(key=np.abs, ascending=False)
)
lr_coefs.head(15)

log_F2A02                      -2.441735
total_expenses__pct3           -2.149040
F2A02__last                    -1.762219
F2A03__last                    -1.332926
endowment_eoy__last            -1.233071
expendable_net_assets__slope   -1.138698
log_F2A03                       1.093052
F2D16__last                    -1.087887
change_in_net_assets__last     -1.083385
log_total_expenses              1.060954
F2D01__last                     1.019215
expendable_net_assets__last    -0.865916
log_F2D16                      -0.825099
total_expenses__last           -0.809483
F2D16__pct3                    -0.795197
dtype: float64

In [42]:
# Persist the per-institution feature matrix for reuse by future notebooks (e.g. once EF lands).
out = feat.copy()
out["closed_by_2024"] = y
out["as_of_year"] = AS_OF
out_path = PROCESSED / f"f2_features_asof{AS_OF}.csv"
out.to_csv(out_path)
out_path, out.shape

(PosixPath('/home/tuna-akin/classes-2026/ann-2026/vigiledu/processed_data/f2_features_asof2019.csv'),
 (1836, 30))